### Preparação ###

#### Importação das bibliotecas ####

In [1]:
from pathlib import Path
import json
import spacy

from dicionario_skills import construir_matchers, extrair_habilidades, classificar, extrair_com_secao, PRIORIDADE
from candidatos import atualizar_fila, propostas, _ler, ARQ_CANDIDATOS

#### Carregando o currículo ####

In [ ]:
curriculo = Path("entrada/curriculo/curriculo.txt").read_text(encoding="utf-8")

#### Carregando as descrições das vagas ####

In [ ]:
pasta_vagas = Path("entrada/vagas_limpas")
vagas = {}

for arquivo in sorted(pasta_vagas.glob("vaga*.txt")):
    vagas[arquivo.name] = arquivo.read_text(encoding="utf-8")

#### Contagem de caracteres nos arquivos carregados ####

In [4]:
print("Currículo: ", len(curriculo), "caracteres")

for nome, texto in vagas.items():
    print(nome, ":", len(texto), "caracteres")

Currículo:  2671 caracteres
vaga1.txt : 4865 caracteres
vaga2.txt : 4639 caracteres
vaga3.txt : 2477 caracteres
vaga4.txt : 3516 caracteres
vaga5.txt : 460 caracteres
vaga6.txt : 1751 caracteres
vaga7.txt : 1744 caracteres
vaga8.txt : 1258 caracteres


### Extração das skills ###

#### Criando os objetos que farão o reconhecimento das skills ####

In [5]:
nlp = spacy.blank("pt") # criando o NLP para dividir o texto em tokens

matcher_skills, matcher_siglas = construir_matchers(nlp) # chamando a função de construir matchers e mostrando o dicionário de skills e siglas que podem aparecer nos currículos e vagas

In [6]:
skills_cv = set(extrair_habilidades(curriculo, nlp, matcher_skills, matcher_siglas)) # chamando a função que procura as skills no currículo e guardando elas na variável 'skills_cv'

print("Habilidades reconhecidas no currículo:", sorted(skills_cv))

Habilidades reconhecidas no currículo: ['análise de dados', 'ciência de dados', 'classificação', 'engenharia de software', 'git', 'inteligência artificial', 'machine learning', 'matplotlib', 'mysql', 'node.js', 'pandas', 'python', 'regressão', 'scikit-learn', 'seaborn']


### Diagnóstico por vaga ###

In [7]:
for nome, texto_vaga in vagas.items():
    skills_vaga = set(extrair_habilidades(texto_vaga, nlp, matcher_skills, matcher_siglas))
    diag = classificar(skills_vaga, skills_cv)

    print(f"\n{nome}: {len(skills_vaga)} requisitos identificados")
    print("  cobertos:", diag["cobertos"])
    for skill, pistas, motivo in diag["inferidos"]:
        print(f"  inferido: {skill} (pistas: {pistas}) - {motivo}")
    for skill, pistas, motivo in diag["relacionados"]:
        print(f"  relacionado: {skill} (pistas: {pistas}) - {motivo}")
    print("  ausentes:", diag["ausentes"])


vaga1.txt: 25 requisitos identificados
  cobertos: ['análise de dados', 'ciência de dados', 'classificação', 'git', 'inteligência artificial', 'machine learning', 'pandas', 'python', 'regressão', 'scikit-learn']
  inferido: análise exploratória (pistas: ['análise de dados', 'ciência de dados']) - análise e ciência de dados com Python envolve análise exploratória
  inferido: numpy (pistas: ['pandas', 'scikit-learn']) - Pandas e Scikit-learn são construídos sobre NumPy
  inferido: overfitting (pistas: ['machine learning']) - conceito básico de qualquer trabalho com Machine Learning
  inferido: sql (pistas: ['mysql']) - MySQL e PostgreSQL são bancos de dados SQL
  relacionado: dashboards (pistas: ['matplotlib', 'seaborn']) - o currículo tem visualização de dados, mas não ferramentas de dashboard/BI
  ausentes: ['api', 'business intelligence', 'engenharia de dados', 'estatística', 'ia generativa', 'llm', 'monitoramento', 'quality assurance', 'rag', 'árvores de decisão']

vaga2.txt: 1 requ

### Relatórios ###

In [11]:
LIMITE_POUCOS = 5
ROTULO = {"REQUISITO": "requisito", "DIFERENCIAL": "diferencial",
          "RESPONSABILIDADE": "responsabilidade", "OUTRO": "outro"}

In [12]:
def relatorio(nome_vaga, texto_vaga):
    secoes = extrair_com_secao(texto_vaga, nlp, matcher_skills, matcher_siglas)
    skills_vaga = set(secoes)
    diag = classificar(skills_vaga, skills_cv)

    soma = sum(len(diag[k]) for k in ("cobertos", "inferidos", "relacionados", "ausentes"))
    assert soma == len(skills_vaga), f"{nome_vaga}: {len(skills_vaga) - soma} requisito(s) sumiram do relatório"

    total = len(skills_vaga)
    atendidos = len(diag["cobertos"]) + len(diag["inferidos"])
    obrigatorios = [s for s in skills_vaga if secoes[s] == "REQUISITO"]
    obrig_ausentes = [s for s in diag["ausentes"] if secoes[s] == "REQUISITO"]

    linhas = [f"# {nome_vaga}", ""]
    if total < LIMITE_POUCOS:
        linhas += [f"> Aviso: só {total} requisito(s) técnico(s) identificado(s). "
                   "O diagnóstico pode não representar a vaga.", ""]

    if total == 0:
        linhas += ["**Nenhum requisito técnico identificado.**", ""]
    else:
        linhas += [f"**Requisitos identificados:** {total}  ",
                   f"**Cobertos ou inferidos:** {atendidos} de {total}  "]
        if obrigatorios:
            linhas += [f"**Obrigatórios ausentes:** {len(obrig_ausentes)} de {len(obrigatorios)}", ""]
        else:
            linhas += ["**Obrigatórios:** a vaga não tem seção de requisitos, então não dá pra separar.", ""]

    linhas.append("## Cobertos")
    linhas += [f"- {s}" for s in diag["cobertos"]] or ["- (nenhum)"]

    linhas += ["", "## Inferidos (o currículo não diz, mas há uma pista quase certa)"]
    linhas += [f"- {s} (pistas: {', '.join(p)}): {m}" for s, p, m in diag["inferidos"]] or ["- (nenhum)"]

    linhas += ["", "## Relacionados (há algo próximo, mas não prova o requisito)"]
    linhas += [f"- {s} (pistas: {', '.join(p)}): {m}" for s, p, m in diag["relacionados"]] or ["- (nenhum)"]

    linhas += ["", "## Ausentes (o currículo não menciona), dos mais importantes pros menos"]
    ordem = sorted(diag["ausentes"], key=lambda s: PRIORIDADE.index(secoes[s]))
    linhas += [f"- {s} ({ROTULO[secoes[s]]})" for s in ordem] or ["- (nenhum)"]

    return "\n".join(linhas)

In [13]:
print(relatorio("vaga1.txt", vagas["vaga1.txt"]))

# vaga1.txt

**Requisitos identificados:** 25  
**Cobertos ou inferidos:** 14 de 25  
**Obrigatórios ausentes:** 2 de 14

## Cobertos
- análise de dados
- ciência de dados
- classificação
- git
- inteligência artificial
- machine learning
- pandas
- python
- regressão
- scikit-learn

## Inferidos (o currículo não diz, mas há uma pista quase certa)
- análise exploratória (pistas: análise de dados, ciência de dados): análise e ciência de dados com Python envolve análise exploratória
- numpy (pistas: pandas, scikit-learn): Pandas e Scikit-learn são construídos sobre NumPy
- overfitting (pistas: machine learning): conceito básico de qualquer trabalho com Machine Learning
- sql (pistas: mysql): MySQL e PostgreSQL são bancos de dados SQL

## Relacionados (há algo próximo, mas não prova o requisito)
- dashboards (pistas: matplotlib, seaborn): o currículo tem visualização de dados, mas não ferramentas de dashboard/BI

## Ausentes (o currículo não menciona), dos mais importantes pros menos
- es

In [14]:
saida = Path("relatorios")
saida.mkdir(exist_ok=True)

for nome, texto in vagas.items():
    (saida / f"{Path(nome).stem}.md").write_text(relatorio(nome, texto), encoding="utf-8")

### Avaliação do sistema ###

In [ ]:
def avaliar(nome_vaga, gabarito):
    texto = vagas[nome_vaga]
    achadas = set(extrair_habilidades(texto, nlp, matcher_skills, matcher_siglas))
    gabarito = set(gabarito)

    acertos = achadas & gabarito
    faltaram = gabarito - achadas
    a_mais = achadas - gabarito

    recall = len(acertos) / len(gabarito) if gabarito else 0
    precisao = len(acertos) / len(achadas) if achadas else 0
    return {"recall": recall, "precisao": precisao,
            "faltaram": sorted(faltaram), "a_mais": sorted(a_mais)}


gabaritos = json.loads(Path("config/gabaritos.json").read_text(encoding="utf-8"))

In [9]:
for grupo, vagas_do_grupo in gabaritos.items():
    print(f"\n== {grupo} ==")
    for nome, gab in vagas_do_grupo.items():
        r = avaliar(nome, gab)
        print(f"{nome}: recall {r['recall']:.0%} | precisão {r['precisao']:.0%}")
        print("  faltaram:", r["faltaram"])
        print("  a mais:  ", r["a_mais"])


== treino ==
vaga1.txt: recall 96% | precisão 92%
  faltaram: ['agentes']
  a mais:   ['engenharia de dados', 'monitoramento']
vaga3.txt: recall 95% | precisão 100%
  faltaram: ['modelagem de dados']
  a mais:   []
vaga4.txt: recall 100% | precisão 95%
  faltaram: []
  a mais:   ['inteligência artificial', 'quality assurance']

== teste ==
vaga6.txt: recall 43% | precisão 100%
  faltaram: ['agentes', 'benchmark', 'gestão de acessos', 'gestão de skills']
  a mais:   []
vaga7.txt: recall 33% | precisão 100%
  faltaram: ['automação de processos', 'excel', 'pacote office', 'visualização de dados']
  a mais:   []


### Fila de candidatos ###

In [10]:
for nome, texto in vagas.items():
    atualizar_fila(nome, texto)

fila = _ler(ARQ_CANDIDATOS, {})
print("termos na fila:", len(fila))

for termo, d in sorted(fila.items(), key=lambda x: len(x[1]["vagas"]), reverse=True)[:20]:
    print(f"{termo} ({len(d['vagas'])} vaga) | {d['exemplo'][:70]}")

termos na fila: 2
excel (1 vaga) | Conhecimento intermediário em Excel;
pacote office (1 vaga) | Conhecimento em Pacote Office;
